# F1 Pit Stop Prediction: Advanced Pipeline

이 노트북은 2026 Kaggle Playground Series (S6E5) 데이터를 활용하여 EDA 인사이트가 반영된 특성 공학(Feature Engineering)과 CatBoost 모델을 활용한 제출 프로세스를 담고 있습니다.

## 1. 환경 설정 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import os
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# Kaggle 데이터 경로 설정
DATA_PATH = '/kaggle/input/playground-series-s6e5/'

train = pd.read_csv(DATA_PATH + 'train.csv')
test = pd.read_csv(DATA_PATH + 'test.csv')
submission = pd.read_csv(DATA_PATH + 'sample_submission.csv')

print(f'Train Shape: {train.shape}, Test Shape: {test.shape}')

## 2. Advanced Feature Engineering
EDA를 통해 얻은 인사이트(타이어별 수명, 경기 후반부 특성, 마모 가속도)를 반영합니다.

In [ ]:
def engineering(df):
    # 1. 타이어 종류별 상대적 수명 (Relative TyreLife)
    compound_mean_life = {
        'HARD': 25, 'MEDIUM': 18, 'SOFT': 12, 'INTERMEDIATE': 20, 'WET': 15
    }
    df['Expected_Life'] = df['Compound'].map(compound_mean_life).fillna(20)
    df['Relative_TyreLife'] = df['TyreLife'] / df['Expected_Life']

    # 2. 경기 후반부 전략 변수 (Is_Final_Laps)
    df['Is_Final_Laps'] = (df['RaceProgress'] > 0.85).astype(int)

    # 3. 마모 가속도 (Degradation Momentum)
    df['Degradation_Momentum'] = df['TyreLife'] * df['Cumulative_Degradation']

    # 4. 물리적 지표 및 변화량
    df['Degradation_per_Lap'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1e-5)
    df['Abs_LapTime_Delta'] = df['LapTime_Delta'].abs()
    
    # 불필요 변수 제거
    df.drop(['Expected_Life'], axis=1, inplace=True)
    return df

train = engineering(train)
test = engineering(test)
print("Feature Engineering Complete.")

## 3. Multi-Model & Ensemble Tournament
각 모델의 학습 과정을 분리하여 실행 및 관리가 용이하도록 구성했습니다.

### 3.1 Data Preparation for Non-CatBoost Models
LGBM과 XGBoost를 위해 범주형 변수를 Label Encoding 합니다.

In [ ]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from scipy.stats import rankdata

train_enc = train.copy()
test_enc = test.copy()
le = LabelEncoder()
cat_features = ['Driver', 'Compound', 'Race', 'Year']
drop_cols = ['id', 'PitNextLap']
features = [c for c in train.columns if c not in drop_cols]

for col in cat_features:
    train_enc[col] = le.fit_transform(train[col].astype(str))
    test_enc[col] = le.transform(test[col].astype(str))

oof_dict = {}
test_dict = {}
kf = GroupKFold(n_splits=5)
X_enc = train_enc[features]
y = train['PitNextLap']
groups = train['Race']
print("Encoding and Setup Complete.")

### 3.2 Model 1: CatBoost Training
범주형 변수를 직접 처리하는 CatBoost를 학습합니다.

In [ ]:
print('--- Training CatBoost ---')
name = 'CatBoost'
model = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, eval_metric='AUC', verbose=0, random_seed=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(kf.split(train, y, groups)):
    X_train, X_val = train.iloc[train_idx][features], train.iloc[val_idx][features]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model.fit(X_train, y_train, cat_features=cat_features, eval_set=(X_val, y_val), early_stopping_rounds=50)
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(test[features])[:, 1] / 5

oof_dict[name] = oof_preds
test_dict[name] = test_preds
print(f'CatBoost OOF AUC: {roc_auc_score(y, oof_preds):.4f}')

### 3.3 Model 2: LightGBM Training

In [ ]:
print('--- Training LightGBM ---')
name = 'LightGBM'
model = LGBMClassifier(n_estimators=1000, learning_rate=0.05, importance_type='gain', random_state=42, verbose=-1)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_enc, y, groups)):
    X_train, X_val = X_enc.iloc[train_idx], X_enc.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model.fit(X_train, y_train)
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(test_enc[features])[:, 1] / 5

oof_dict[name] = oof_preds
test_dict[name] = test_preds
print(f'LightGBM OOF AUC: {roc_auc_score(y, oof_preds):.4f}')

### 3.4 Model 3: XGBoost Training

In [ ]:
print('--- Training XGBoost ---')
name = 'XGBoost'
model = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=6, eval_metric='auc', random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_enc, y, groups)):
    X_train, X_val = X_enc.iloc[train_idx], X_enc.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model.fit(X_train, y_train)
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(test_enc[features])[:, 1] / 5

oof_dict[name] = oof_preds
test_dict[name] = test_preds
print(f'XGBoost OOF AUC: {roc_auc_score(y, oof_preds):.4f}')

### 3.5 Ensemble & Strategy Comparison
모든 모델의 결과를 조합하여 최적의 앙상블 전략을 도출합니다.

In [ ]:
print('\n--- Calculating Ensembles ---')
oof_dict['Simple_Avg'] = (oof_dict['CatBoost'] + oof_dict['LightGBM'] + oof_dict['XGBoost']) / 3
test_dict['Simple_Avg'] = (test_dict['CatBoost'] + test_dict['LightGBM'] + test_dict['XGBoost']) / 3

oof_dict['Weighted_Avg'] = (oof_dict['CatBoost'] * 0.5) + (oof_dict['LightGBM'] * 0.3) + (oof_dict['XGBoost'] * 0.2)
test_dict['Weighted_Avg'] = (test_dict['CatBoost'] * 0.5) + (test_dict['LightGBM'] * 0.3) + (test_dict['XGBoost'] * 0.2)

oof_dict['Rank_Ensemble'] = (rankdata(oof_dict['CatBoost']) + rankdata(oof_dict['LightGBM']) + rankdata(oof_dict['XGBoost'])) / 3
test_dict['Rank_Ensemble'] = (rankdata(test_dict['CatBoost']) + rankdata(test_dict['LightGBM']) + rankdata(test_dict['XGBoost'])) / 3

final_results = []
for name, preds in oof_dict.items():
    final_results.append({'Strategy': name, 'AUC': roc_auc_score(y, preds)})

results_df = pd.DataFrame(final_results).sort_values(by='AUC', ascending=False)
print(results_df)

## 4. 최종 결과 저장 및 제출
가장 높은 성능을 보인 전략을 선택하여 최종 제출 파일을 생성합니다.

In [ ]:
best_strategy = results_df.iloc[0]['Strategy']
print(f'Final Chosen Strategy: {best_strategy}')

submission['PitNextLap'] = test_dict[best_strategy]
submission.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' saved.")

del train, test, train_enc, test_enc, X_enc, y; gc.collect()